# 04 — Reaction Prevalence: Growers vs Non-growers

In [1]:
from pathlib import Path
import sys

# Find the project root (parent of notebooks/) so absolute paths work
# from anywhere the notebook is launched.
PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
REPORTS = PROJECT_ROOT / 'reports'
RESULTS = PROJECT_ROOT / 'results'
LOGS    = PROJECT_ROOT / 'logs'
DATA    = PROJECT_ROOT / 'data'
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))

# KBUtils NotebookSession: opens .kbcache/ alongside this notebook so
# heavy outputs can be saved/loaded with provenance.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(notebook_file=__file__ if '__file__' in dir() else None,
                                       project_name='core_models_analysis')
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


## What this notebook does

Reads every model JSON, extracts the set of `seed.reaction`
annotations, and computes the prevalence delta between growers and
non-growers — mirroring `scripts/deeper_analysis.py` part 3. The
per-model reaction set is cached so the diversity-selection notebook
can reuse it without re-reading 5,683 files.

In [2]:
import json, pandas as pd, multiprocessing as mp
from collections import Counter
from pathlib import Path

df = pd.read_csv(RESULTS / 'results.csv')
df['grows'] = df['grows'].astype(str).str.lower() == 'true'
MODELS = DATA / 'core_models_kegg2'

def extract(mid):
    with open(MODELS / f'{mid}.json') as f:
        m = json.load(f)
    return mid, {r['annotation'].get('seed.reaction')
                 for r in m['reactions']
                 if r.get('annotation', {}).get('seed.reaction')}

# Reuse cache if we ran this before
try:
    cached = session.cache.load('rxnsets_by_model')
    print(f'loaded {len(cached)} reaction sets from cache')
    rxnsets = {mid: set(rxns) for mid, rxns in cached.items()}
except KeyError:
    print('cache miss — extracting from JSON (may take ~30s)…')
    with mp.Pool(16) as pool:
        rxnsets = dict(pool.imap_unordered(extract, df['model_id'], chunksize=8))
    # save as plain dict of lists for JSON serializer
    session.cache.save('rxnsets_by_model',
                       {mid: sorted(s) for mid, s in rxnsets.items()},
                       type_hint='json',
                       metadata={'n_models': len(rxnsets)})
    print(f'cached {len(rxnsets)} reaction sets')

cache miss — extracting from JSON (may take ~30s)…


cached 5683 reaction sets


In [3]:
growers = set(df.loc[df['grows'], 'model_id'])
nongrow = set(df.loc[~df['grows'], 'model_id'])
gC, nC = Counter(), Counter()
for mid, rs in rxnsets.items():
    tgt = gC if mid in growers else nC
    tgt.update(rs)

all_r = set(gC) | set(nC)
rows = []
for r in all_r:
    gf, nf = gC[r] / len(growers), nC[r] / len(nongrow)
    rows.append((r, gC[r], 100*gf, nC[r], 100*nf, 100*(gf - nf)))
prev_df = pd.DataFrame(rows, columns=['rxn', 'g_count', 'g_pct',
                                      'n_count', 'n_pct', 'delta_pct'])
prev_df.sort_values('delta_pct', ascending=False).head(15)

,rxn,g_count,g_pct,n_count,n_pct,delta_pct
85,rxn05561,2775,80.179139,388,17.461746,62.717393
11,rxn00330,2371,68.506212,173,7.785779,60.720433
239,rxn00006,3178,91.823172,792,35.643564,56.179608
78,rxn00336,2189,63.247616,176,7.920792,55.326824
53,rxn02167,2860,82.635077,611,27.497750,55.137327
114,rxn03250,2842,82.114996,604,27.182718,54.932277
137,rxn03240,2842,82.114996,604,27.182718,54.932277
95,rxn02376,3032,87.604739,756,34.023402,53.581336
207,rxn00441,3032,87.604739,756,34.023402,53.581336
134,rxn01872,3023,87.344698,758,34.113411,53.231287


In [4]:
prev_df.sort_values('delta_pct').head(15)   # most enriched in NON-growers

,rxn,g_count,g_pct,n_count,n_pct,delta_pct
182,rxn05466_c,418,12.077434,1067,48.019802,-35.942368
101,rxn05319_c,1865,53.886160,1780,80.108011,-26.221851
154,rxn05759,102,2.947125,396,17.821782,-14.874657
153,rxn05488_c,2832,81.826062,2093,94.194419,-12.368358
42,rxn13974,964,27.853222,891,40.099010,-12.245788
41,rxn00782,54,1.560243,304,13.681368,-12.121125
91,rxn02527,261,7.541173,434,19.531953,-11.990780
169,rxn05939,920,26.581913,834,37.533753,-10.951841
178,rxn05559_c,2455,70.933256,1805,81.233123,-10.299867
89,rxn40505,57,1.646923,262,11.791179,-10.144256


---
## Report: `reports/REACTION_PREVALENCE.md`

# Part 3 — Reaction prevalence: growers vs non-growers

Growers: 3461    Non-growers: 2222

Each row is a `seed.reaction` ID; fractions = share of models containing it.

## Top 30 reactions most enriched in GROWERS
| seed.reaction | grower count | grower % | non-grower count | non-grower % | Δ |
|---|---|---|---|---|---|
| rxn05561 | 2775 | 80.2% | 388 | 17.5% | +62.7% |
| rxn00330 | 2371 | 68.5% | 173 | 7.8% | +60.7% |
| rxn00006 | 3178 | 91.8% | 792 | 35.6% | +56.2% |
| rxn00336 | 2189 | 63.2% | 176 | 7.9% | +55.3% |
| rxn02167 | 2860 | 82.6% | 611 | 27.5% | +55.1% |
| rxn03250 | 2842 | 82.1% | 604 | 27.2% | +54.9% |
| rxn03240 | 2842 | 82.1% | 604 | 27.2% | +54.9% |
| rxn02376 | 3032 | 87.6% | 756 | 34.0% | +53.6% |
| rxn00441 | 3032 | 87.6% | 756 | 34.0% | +53.6% |
| rxn01872 | 3023 | 87.3% | 758 | 34.1% | +53.2% |
| rxn00379 | 2995 | 86.5% | 767 | 34.5% | +52.0% |
| rxn03249 | 2865 | 82.8% | 701 | 31.5% | +51.2% |
| rxn06777 | 2865 | 82.8% | 701 | 31.5% | +51.2% |
| rxn09003 | 2643 | 76.4% | 574 | 25.8% | +50.5% |
| rxn05937 | 2342 | 67.7% | 383 | 17.2% | +50.4% |
| rxn00175 | 3108 | 89.8% | 883 | 39.7% | +50.1% |
| rxn01480 | 2629 | 76.0% | 583 | 26.2% | +49.7% |
| rxn09001 | 2600 | 75.1% | 590 | 26.6% | +48.6% |
| rxn10126 | 2644 | 76.4% | 627 | 28.2% | +48.2% |
| rxn14427 | 2662 | 76.9% | 641 | 28.8% | +48.1% |
| rxn00199 | 3412 | 98.6% | 1123 | 50.5% | +48.0% |
| rxn01387 | 3412 | 98.6% | 1123 | 50.5% | +48.0% |
| rxn00623 | 2152 | 62.2% | 317 | 14.3% | +47.9% |
| rxn00083 | 2397 | 69.3% | 478 | 21.5% | +47.7% |
| rxn09295 | 2397 | 69.3% | 478 | 21.5% | +47.7% |
| rxn05627 | 2668 | 77.1% | 652 | 29.3% | +47.7% |
| rxn14414 | 2397 | 69.3% | 484 | 21.8% | +47.5% |
| rxn14412 | 2397 | 69.3% | 484 | 21.8% | +47.5% |
| rxn00371 | 2345 | 67.8% | 458 | 20.6% | +47.1% |
| rxn00256 | 3423 | 98.9% | 1179 | 53.1% | +45.8% |

## Top 30 reactions most enriched in NON-GROWERS
| seed.reaction | grower count | grower % | non-grower count | non-grower % | Δ |
|---|---|---|---|---|---|
| rxn05466_c | 418 | 12.1% | 1067 | 48.0% | -35.9% |
| rxn05319_c | 1865 | 53.9% | 1780 | 80.1% | -26.2% |
| rxn05759 | 102 | 2.9% | 396 | 17.8% | -14.9% |
| rxn05488_c | 2832 | 81.8% | 2093 | 94.2% | -12.4% |
| rxn13974 | 964 | 27.9% | 891 | 40.1% | -12.2% |
| rxn00782 | 54 | 1.6% | 304 | 13.7% | -12.1% |
| rxn02527 | 261 | 7.5% | 434 | 19.5% | -12.0% |
| rxn05939 | 920 | 26.6% | 834 | 37.5% | -11.0% |
| rxn05559_c | 2455 | 70.9% | 1805 | 81.2% | -10.3% |
| rxn40505 | 57 | 1.6% | 262 | 11.8% | -10.1% |
| rxn05469_c | 3121 | 90.2% | 2221 | 100.0% | -9.8% |
| rxn03644 | 320 | 9.2% | 374 | 16.8% | -7.6% |
| rxn03643 | 292 | 8.4% | 352 | 15.8% | -7.4% |
| rxn15962 | 12 | 0.3% | 165 | 7.4% | -7.1% |
| rxn00151 | 1021 | 29.5% | 805 | 36.2% | -6.7% |
| rxn00869 | 1209 | 34.9% | 912 | 41.0% | -6.1% |
| rxn03126 | 23 | 0.7% | 119 | 5.4% | -4.7% |
| rxn03085 | 0 | 0.0% | 102 | 4.6% | -4.6% |
| rxn03127 | 0 | 0.0% | 95 | 4.3% | -4.3% |
| rxn03020 | 0 | 0.0% | 92 | 4.1% | -4.1% |
| rxn00157 | 1076 | 31.1% | 782 | 35.2% | -4.1% |
| rxn00172 | 518 | 15.0% | 423 | 19.0% | -4.1% |
| rxn04794 | 1069 | 30.9% | 770 | 34.7% | -3.8% |
| rxn07191 | 2 | 0.1% | 84 | 3.8% | -3.7% |
| rxn00162 | 387 | 11.2% | 323 | 14.5% | -3.4% |
| rxn00871 | 10 | 0.3% | 80 | 3.6% | -3.3% |
| rxn01236 | 1 | 0.0% | 67 | 3.0% | -3.0% |
| rxn15383 | 626 | 18.1% | 462 | 20.8% | -2.7% |
| rxn02480 | 116 | 3.4% | 131 | 5.9% | -2.5% |
| sul00003 | 60 | 1.7% | 87 | 3.9% | -2.2% |